# Kraków Micromobility Analysis with Polars

Exploratory analysis of the Gold marts produced by the dbt/DuckDB pipeline for the **Dott Kraków** shared e-scooter system (GBFS feed).

**Prerequisite:** run `python ingestion/fetch_gbfs.py --loop 60` for a while, then `dbt run`.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

## 1. Load Gold marts (lazy)

In [ ]:
utilization = pl.scan_parquet("../dbt-project/data/gold/mart_station_utilization_hourly.parquet")
fleet = pl.scan_parquet("../dbt-project/data/gold/mart_fleet_health_hourly.parquet")
trips = pl.scan_parquet("../dbt-project/data/gold/mart_trips_enriched.parquet")

utilization.collect_schema(), fleet.collect_schema(), trips.collect_schema()

## 2. Trip characteristics

In [ ]:
trip_stats = trips.select(
    pl.col("duration_min").mean().alias("avg_duration_min"),
    pl.col("distance_m").mean().alias("avg_distance_m"),
    pl.col("fare_pln").mean().alias("avg_fare_pln"),
    pl.len().alias("trip_segments"),
).collect()
trip_stats

## 3. Fleet battery health over time

In [ ]:
battery = (
    fleet
    .group_by("hour")
    .agg(
        (pl.col("avg_battery_pct") * pl.col("observations")).sum().alias("w"),
        pl.col("observations").sum().alias("n"),
        (pl.col("low_battery_share") * pl.col("observations")).sum().alias("low_w"),
    )
    .with_columns(
        (pl.col("w") / pl.col("n")).alias("fleet_avg_battery"),
        (pl.col("low_w") / pl.col("n")).alias("low_battery_share"),
    )
    .sort("hour")
    .collect()
)

fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.plot(battery["hour"], battery["fleet_avg_battery"], color="tab:green", label="avg battery %")
ax1.set_ylabel("Avg battery %")
ax2 = ax1.twinx()
ax2.plot(battery["hour"], battery["low_battery_share"], color="tab:red", label="low battery share")
ax2.set_ylabel("Share < 20% battery")
ax1.set_title("Fleet battery health over time")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 4. Stations that run dry most often

In [ ]:
driest = (
    utilization
    .group_by("station_id")
    .agg(pl.col("empty_share").mean().alias("empty_share"))
    .sort("empty_share", descending=True)
    .limit(15)
    .collect()
)
driest

## 5. Revenue simulation from real pricing plans

In [ ]:
revenue = (
    trips
    .with_columns(pl.col("departed_at").dt.hour().alias("hour_of_day"))
    .group_by("hour_of_day")
    .agg(
        pl.len().alias("rides"),
        pl.col("fare_pln").sum().alias("revenue_pln"),
    )
    .sort("hour_of_day")
    .collect()
)

plt.figure(figsize=(10, 4))
plt.bar(revenue["hour_of_day"], revenue["revenue_pln"])
plt.title("Estimated revenue by hour of day (operator pricing)")
plt.xlabel("Hour")
plt.ylabel("PLN")
plt.tight_layout()
plt.show()